# Gold — Top Resources

Rankeia os recursos mais caros por mês.

**Métricas:**
- `total_cost_usd` — custo total do recurso no mês
- `rank` — posição no ranking mensal (1 = mais caro)

**Top 20 por mês.** KPI: Top recursos mais caros.

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import sys
import os

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if __import__("os").path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session, postgres_jdbc_url, postgres_props
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DecimalType, TimestampType, DateType
)


In [ ]:
from pyspark.sql.window import Window

spark = create_spark_session("gold_top_resources")

df_entries = spark.read.format("delta").load("s3a://datalake/silver/cost_entries/")

df_agg = (
    df_entries
    .groupBy("resource_name", "resource_type", "team_name", "provider", "environment", "year", "month")
    .agg(F.round(F.sum("cost_usd"), 4).alias("total_cost_usd"))
)

window_monthly = Window.partitionBy("year", "month").orderBy(F.col("total_cost_usd").desc())

df_gold = (
    df_agg
    .withColumn("rank", F.rank().over(window_monthly))
    .filter(F.col("rank") <= 20)
    .withColumn("_processed_at", F.current_timestamp())
    .orderBy("year", "month", "rank")
)

output_path = "s3a://datalake/gold/top_resources/"
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(output_path)
)

print(f"[gold_top_resources] {df_gold.count()} linhas → {output_path}")
df_gold.select(
    "year", "month", "rank", "resource_name", "resource_type", "team_name", "provider", "total_cost_usd"
).show(10, truncate=False)
spark.stop()
